# 03 — Fare and pricing features

**Goal:** test whether price / price-change / fare-per-km features improve ROC-AUC.

## Why this experiment?
`hyper_ack` is likely related to whether a courier accepts a job.
Price, price vs distance, and how much the fare changed may be strong signals.

## Approach
1. Start from baseline numeric features.
2. Add pricing transforms: fare-per-km, final−first delta, customer/biker gap, log fares.
3. Train the same LightGBM setup as experiment 02.
4. Compare ROC-AUC against experiment 02.

## What changed?
- Features only: added pricing-derived columns
- Model stays LightGBM (same family as 02)
- **Caution:** `final_*` fares may be available only after the decision (see experiment 14)

## Features used in this notebook
- Base raw columns
- **New pricing features:** fare-per-km, fare deltas, log fares, biker-customer gap
- **How selected:** business logic — couriers care about pay vs trip effort and price changes
- **Why:** pricing was one of the strongest single FE ideas in the suite


### Setup
Shared data load and fixed split.


In [ ]:
import os
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate
        break
    if (candidate / "hyperack_exp" / "shared" / "protocol.py").exists():
        EXPERIMENT_ROOT = candidate / "hyperack_exp"
        break
else:
    raise RuntimeError("Run this notebook from the HyperAck project directory.")
os.chdir(EXPERIMENT_ROOT)
sys.path.insert(0, str(EXPERIMENT_ROOT))

from shared.protocol import (
    add_geo_features,
    add_pricing_features,
    add_time_features,
    base_features,
    evaluate,
    load_clean_df,
    make_xy,
    save_result,
    split_frame,
    actual_vs_predicted_report,
)

RANDOM_STATE = 42
train_df, test_df = split_frame(load_clean_df())


### Feature engineering
Add fare ratios, deltas, and log transforms.


In [ ]:
X_train, y_train = make_xy(train_df)
X_test, y_test = make_xy(test_df)
X_train = add_pricing_features(X_train)
X_test = add_pricing_features(X_test)


### Features selected and why

We keep the base columns and **add pricing transforms**.

| New feature | How made | Why important |
|-------------|----------|---------------|
| first_fare_per_km | first fare ÷ distance | Pay attractiveness per km |
| customer_fare_delta | final − first | Did price move? |
| customer_fare_change_pct | delta ÷ first | Relative change |
| biker_customer_gap | biker − customer | Margin / split |
| log_* / *_per_km | log or ratio forms | Stabilize scale / outliers |

**Caution:** features from `final_*` fares may not exist before the decision (see notebook 14).

Next cell lists every selected column.


In [ ]:
feature_why = {
    "deliverey_category_id": "Delivery type — some categories get accepted more often",
    "weekday": "Day of week — weekday vs weekend courier behavior",
    "time_bucket": "Coarse time-of-day bucket from the raw data",
    "total_distance": "Trip length — longer trips can be harder to accept",
    "sum_product": "Order size / number of products",
    "source_latitude": "Pickup latitude — area effects",
    "source_longitude": "Pickup longitude — area effects",
    "destination_latitude": "Drop-off latitude — area effects",
    "destination_longitude": "Drop-off longitude — area effects",
    "first_customer_fare": "First offered customer price (usually known early)",
    "final_customer_fare": "Final customer price — strong but may be post-decision",
    "final_biker_fare": "Final courier pay — strong but may be post-decision",
    "geo_cluster": "Train-only KMeans region of the trip (pickup+drop-off)",
    "log_distance": "Log distance — softens very long trips",
    "first_fare_per_km": "First fare ÷ distance — pay vs effort",
    "final_customer_fare_per_km": "Final customer fare ÷ distance",
    "customer_fare_delta": "Final − first customer fare (price change)",
    "customer_fare_change_pct": "Relative fare change vs first offer",
    "biker_customer_gap": "Biker fare − customer fare (split / margin)",
    "biker_fare_per_km": "Courier pay per km",
    "log_final_customer_fare": "Log of final customer fare",
    "log_final_biker_fare": "Log of final biker fare",
    "hour": "Exact hour of order creation",
    "is_rush_hour": "Lunch/evening peak flag",
    "is_weekend": "Weekend flag",
    "hour_sin": "Cyclical hour (sin) so 23 is near 0",
    "hour_cos": "Cyclical hour (cos)",
    "weekday_sin": "Cyclical weekday (sin)",
    "weekday_cos": "Cyclical weekday (cos)",
    "day_of_month": "Calendar day — mild monthly pattern",
    "haversine_km": "Great-circle route distance in km",
    "latitude_delta": "North/south trip span",
    "longitude_delta": "East/west trip span",
    "geo_bearing_sin": "Trip direction (sin of bearing)",
    "geo_bearing_cos": "Trip direction (cos of bearing)",
    "distance_x_first_fare": "Interaction: long trip × price",
    "category_x_hour": "Interaction: category × hour",
    "total_distance_qbin": "Train-fitted distance quantile bin",
    "first_customer_fare_qbin": "Train-fitted first-fare quantile bin"
}

cols = list(X_train.columns)
rows = []
for c in cols:
    rows.append({
        "feature": c,
        "why_selected": feature_why.get(c, "Part of this experiment's engineered feature set"),
    })
feature_table = pd.DataFrame(rows)
print(f"Total features selected: {len(cols)}")
print("Columns:")
print(", ".join(cols))
feature_table


### Model
Same LightGBM setup; only features changed.


In [ ]:
from lightgbm import LGBMClassifier
model = Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", LGBMClassifier(n_estimators=700, learning_rate=0.04, num_leaves=31, reg_lambda=1.0, random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1))])


### Evaluate
Held-out metrics.


In [ ]:
metrics = evaluate(model, X_train, y_train, X_test, y_test)
metrics


### Actual vs predicted (test set)

After training, we score the **held-out test set** and compare:

1. **Actual** labels (`hyper_ack`) vs **predicted** labels  
2. Confusion matrix (rows = actual, columns = predicted)  
3. Per-class precision / recall / F1  
4. A sample of correct and incorrect rows with predicted probability  

This is only test-set performance — not training rows.


In [ ]:
from IPython.display import display
from shared.protocol import actual_vs_predicted_report

avp = actual_vs_predicted_report(
    metrics["y_true"],
    metrics["y_pred"],
    metrics["y_prob"],
    sample_size=25,
)
print("1) Actual vs predicted class counts")
display(avp["class_counts"])
print("2) Confusion matrix")
display(avp["confusion_matrix"])
print("3) Outcome breakdown")
display(avp["outcomes"])
print("4) Per-class metrics")
display(avp["per_class_metrics"])
print("5) Sample of actual vs predicted rows")
display(avp["prediction_sample"])


### Save result
Write experiment `03`.


In [ ]:
result_path = save_result(
    "03",
    "03_fare_pricing_features",
    "LightGBM with fare, fare-delta, and fare-per-km features",
    metrics,
    best_model="LGBMClassifier",
    notes="Pricing features may be highly predictive but can be post-decision leakage.",
    feature_count=X_train.shape[1],
)
pd.Series(metrics).drop("confusion_matrix").sort_index(), result_path


## What to look at
- Did pricing features lift ROC-AUC vs 02?
- If the lift is huge, check experiment 14 (leakage-safe) before trusting it.
